In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
N   0.000   0.000   1.395
N   0.000   0.000  -1.395
C   0.000   1.133   0.697
C   0.000  -1.133   0.697
C   0.000  -1.133  -0.697
C   0.000   1.133  -0.697
H   0.000   2.063   1.252
H   0.000  -2.063   1.252
H   0.000  -2.063  -1.252
H   0.000   2.063  -1.252'''
molecule = 'p_C4H4N2'
spin = 0
charge =  0


In [3]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [5]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./p_C4H4N2.txt
orbital  energy   occ
0     -15.571884488619505    2.0
1     -15.571855909987676    2.0
2     -11.247410544341207    2.0
3     -11.247239970061782    2.0
4     -11.246723926391631    2.0
5     -11.246694767368647    2.0
6     -1.2666860527756811    2.0
7     -1.1729450863067001    2.0
8     -1.0070218294956694    2.0
9     -0.8504548382324996    2.0
10     -0.8325396047870544    2.0
11     -0.6849430523531755    2.0
12     -0.6386166731223129    2.0
13     -0.6362307211974391    2.0
14     -0.5710293348789087    2.0
15     -0.5337064251265732    2.0
16     -0.49647935771705987    2.0
17     -0.42223298588949065    2.0
18     -0.37392704817873684    2.0
19     -0.32783525963339333    2.0
20     -0.32440007847548497    2.0
21     0.21260157208032482    0.0
22     0.22008865818680123    0.0
23     0.45470396505418154    0.0
24     0.6061198992379409    0.0
25     0.6235064674827709    0.0
26     0.6478788104093112    0.0
27     0.6671961898823963    0.0
28     

In [7]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([16,19,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-262.03618528970975),
 np.float64(-7.050413262900037),
 FCIvector([[ 9.13261805e-01,  0.00000000e+00,  0.00000000e+00,
             -4.79692083e-03,  0.00000000e+00,  0.00000000e+00,
              0.00000000e+00, -8.33482325e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -1.17575526e-03,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
             -2.09830842e-02,  0.00000000e+00, -2.32763642e-02,
              0.00000000e+00,  0.00000000e+00],
            [ 0.00000000e+00, -2.61895057e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -1.14881405e-01,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
             -1.03786937e-02,  0.00000000e+00,  0.00000000e+00,
              0.00000000e+00,  9.54887368e-03,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00, -1.78734033e-01,
     

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.913261804818273
[0 1 2]      [1 2 3]      -0.004796920833870
[0 1 2]      [0 3 4]      -0.083348232491144
[0 1 2]      [0 2 5]      -0.001175755256968
[0 1 2]      [2 3 5]      -0.020983084161375
[0 1 2]      [1 4 5]      -0.023276364225647
[0 1 3]      [0 1 3]      -0.026189505692941
[0 1 3]      [0 2 4]      -0.114881405398549
[0 1 3]      [2 3 4]      -0.010378693738453
[0 1 3]      [0 3 5]      0.009548873680376
[0 2 3]      [0 2 3]      -0.178734032533092
[0 2 3]      [0 1 4]      -0.031533243313551
[0 2 3]      [1 3 4]      -0.010302379486120
[0 2 3]      [1 2 5]      0.060306183528914
[0 2 3]      [0 4 5]      0.009533475610068
[0 2 3]      [3 4 5]      -0.008379279700000
[1 2 3]      [0 1 2]      -0.004796920833870
[1 2 3]      [1 2 3]      -0.050033413749767
[1 2 3]      [0 3 4]      0.000076168217512
[1 2 3]      [0 2 5]      0.081289232351261
[1 2 3]      [2 3 5]      0.007151329269002
[1 2 3]      [1 4 5]      -0.005701146365449
[0 1 4]      [0 2